# Predictive Employee Attrition Modelling

## Objectives


The aim of this notebook is to build a supervised machine learning model to predict employee attrition and identify the features most strongly associated with employees leaving the organisation.

This supports the business requirement of helping HR teams take proactive retention measures by identifying employees at higher risk of attrition.

The target variable for this analysis is `Attrition`.



# Input

cleanser file data\processed\attrition_clean_v1.csv

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

In [ ]:
# Imports

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# notebook display settings
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## Load and Review Modelling Dataset

In [ ]:
#Load cleaned dataset from 01_ETL.ipynb

file_path = current_dir + "/data/processed/attrition_clean_v1.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())

The cleaned dataset from the ETL stage is loaded and reviewed to confirm its structure before modelling.

## Review Target Variable

In [ ]:
print("Attrition value counts:")
print(df["Attrition"].value_counts())
print("\nAttrition percentage distribution:")
print(round(df["Attrition"].value_counts(normalize=True) * 100, 2))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Attrition")
plt.title("Attrition Distribution")
plt.xlabel("Attrition")
plt.ylabel("Count")
plt.show()

The target variable was reviewed before modelling to understand class balance. Attrition cases are less common than non-attrition cases, so accuracy alone may give a misleading view of performance. For this reason, additional metrics such as precision, recall, F1-score, and ROC AUC were used

## Define Features and Target

In [ ]:
# Create a working copy
df_model = df.copy()

# Convert target variable to binary
# Adjust this mapping only if your Attrition column is already numeric
y = df_model["Attrition"].map({"Yes": 1, "No": 0})

# Drop target and any ID-like fields that should not be used for prediction
X = df_model.drop(columns=["Attrition", "EmployeeNumber"], errors="ignore")

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nTarget preview:")
display(y.head())

The target variable was converted into binary form, where `Yes = 1` and `No = 0`, to support binary classification modelling.

## Identify Numeric and Categorical Features

In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\ny_train distribution:")
print(y_train.value_counts(normalize=True))
print("\ny_test distribution:")
print(y_test.value_counts(normalize=True))

The dataset was split into training and test sets using stratification so that the attrition distribution remained similar in both subsets. This supports a fairer evaluation of model performance.

## Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

A preprocessing pipeline was created to scale numeric variables and one-hot encode categorical variables. This ensures the data is transformed consistently before being passed into the model.

## Baseline Model: Logistic Regression

In [ ]:
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    ))
])

log_reg_pipeline.fit(X_train, y_train)

y_pred = log_reg_pipeline.predict(X_test)
y_prob = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("Model training complete.")

Logistic Regression was selected as a baseline model because it is appropriate for binary classification and provides interpretable coefficients, making it suitable for understanding which factors are most strongly associated with attrition.

## Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression Performance")
print("-" * 35)
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 Score : {f1:.3f}")
print(f"ROC AUC  : {roc_auc:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

The confusion matrix shows that the model correctly identified 31 out of 47 employees who left, giving a recall of 66.0%. It also correctly classified 191 employees who stayed. However, the model produced 56 false positives, meaning some employees were flagged as at risk despite not leaving. This indicates that the model is more useful as an early warning tool than as a standalone decision-making system.

In [ ]:
plt.figure(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve - Logistic Regression")
plt.show()

The model was evaluated using accuracy, precision, recall, F1-score, and ROC AUC. For this business problem, recall is especially important because failing to identify employees at risk of leaving could reduce the usefulness of the model for proactive retention planning.

In [ ]:
# Get feature names after preprocessing
ohe = log_reg_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
encoded_cat_features = ohe.get_feature_names_out(categorical_features)

all_feature_names = np.concatenate([numeric_features, encoded_cat_features])

# Get coefficients
coefficients = log_reg_pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Coefficient": coefficients
})

coef_df = coef_df.sort_values("Coefficient", ascending=False)

display(coef_df.head(10))
display(coef_df.tail(10))

The Logistic Regression coefficients were reviewed to identify which features were associated with increased or decreased attrition likelihood. Positive coefficients indicate a higher predicted likelihood of attrition, while negative coefficients indicate a lower predicted likelihood. These values should be interpreted relative to the reference category used during one-hot encoding.

In [ ]:

top_positive = coef_df.head(10)
top_negative = coef_df.tail(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_positive, x="Coefficient", y="Feature")
plt.title("Top Positive Drivers of Attrition")
plt.show()


The strongest positive predictors of attrition included overtime, frequent business travel, selected job roles, being single, a longer time since last promotion, and a higher number of companies previously worked for. These factors were associated with a greater predicted likelihood of employees leaving the organisation.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=top_negative, x="Coefficient", y="Feature")
plt.title("Top Negative Drivers of Attrition")
plt.show()

Several features were associated with a lower predicted likelihood of attrition, including job satisfaction, environment satisfaction, years with current manager, total working years, and certain job roles. These factors appear to be linked with stronger employee retention within the dataset.

Coefficients should be interpreted carefully because some features may be correlated with each other. For example, variables such as job level, total working years, and promotion history may overlap in the information they capture, which can influence coefficient size and direction.

The model suggests that overtime, frequent business travel, selected job roles, and time since last promotion are associated with a greater likelihood of attrition. Some coefficients, such as job level, should be interpreted cautiously because correlated variables may influence coefficient direction

## Conclusion


A supervised machine learning approach was used to predict employee attrition. The Logistic Regression model showed good discriminatory ability overall, with an ROC AUC of 0.803, and identified around two-thirds of employees who left. While the model generated a relatively high number of false positives, it still provides useful value as an early warning tool for HR. The coefficient analysis also highlighted several factors associated with attrition risk, including overtime, business travel, and time since last promotion.
 Overall, the model supports the business objective of identifying employees at higher risk of leaving so that proactive retention measures can be considered.

### Limitations
- The dataset may contain class imbalance, which can affect performance interpretation.
- Correlation does not necessarily imply causation.
- Model performance is constrained by the features available in the dataset.
- Further tuning or additional models could improve predictive accuracy.

### Next Steps
- Tune the best-performing model
- Explore feature selection
- Integrate key insights into the dashboard/app
- Present high-risk employee patterns in a business-friendly way